# Trabalho Prático 06 — Agentes Inteligentes
## Assistente de Acessibilidade: avaliação de contraste

**Disciplina:** GCC128 — Inteligência Artificial  
**Framework:** LangGraph | **LLM:** Groq (plano gratuito)  

**Objetivo:** analisar o contraste entre texto e fundo de um componente de interface e recomendar melhorias.

**Agentes**
1. **Auditor de Acessibilidade:** usa a ferramenta `avaliar_contraste` e explica o diagnóstico.
2. **Consultor UX:** recebe a análise e propõe melhorias sem alterar os resultados matemáticos.

**Fluxo:** Entrada → Auditor (+ ferramenta) → Consultor UX → Relatório.

> O notebook exige internet e uma chave gratuita do Groq para executar as células que usam LLM. A ferramenta de contraste pode ser testada sem chave.

## 1. Instalação
Execute a célula abaixo uma vez no ambiente Jupyter ou Google Colab.

In [12]:
pip install -q -U langgraph langchain-groq langchain-core

Note: you may need to restart the kernel to use updated packages.


## 2. Imports e chave de acesso
Crie uma chave no console do Groq e execute esta célula. No VS Code, digite/cole a chave no campo de entrada que aparece abaixo da célula e confirme com Enter; a entrada fica oculta. A chave também pode ser definida na variável de ambiente `GROQ_API_KEY`. **Não cole a chave no código nem a inclua no ZIP.**

In [13]:
import os
from getpass import getpass
from typing import TypedDict
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END

# A variável de ambiente é opcional; sem ela, a chave será solicitada sem exibição.
groq_api_key = os.environ.get("GROQ_API_KEY", "").strip()
if not groq_api_key:
    groq_api_key = getpass("Cole sua chave de API do Groq (entrada oculta): ").strip()

if not groq_api_key:
    raise ValueError(
        "A chave do Groq não foi informada. Execute esta célula novamente e cole a chave "
        "secreta criada em https://console.groq.com/keys."
    )

os.environ["GROQ_API_KEY"] = groq_api_key

# Modelo substituto recomendado pelo Groq para llama-3.3-70b-versatile.
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0, api_key=groq_api_key)
print("Chave recebida e LLM configurado.")

Chave recebida e LLM configurado.


## 3. Ferramenta `@tool`: contraste WCAG

A ferramenta faz um cálculo **determinístico** (não depende do LLM). Calcula a luminância relativa conforme sRGB e a razão de contraste. O limiar AA é **4,5:1 para texto normal** e **3:1 para texto grande**. Esta implementação avalia **somente contraste de texto**, não substitui uma auditoria WCAG completa.

In [14]:
def _hex_rgb(cor: str) -> tuple[int, int, int]:
    cor = cor.strip().lstrip("#")
    if len(cor) != 6 or any(c not in "0123456789abcdefABCDEF" for c in cor):
        raise ValueError("Informe cores em HEX de 6 dígitos, por exemplo #FFFFFF.")
    return tuple(int(cor[i:i+2], 16) for i in (0, 2, 4))


def _luminancia(cor: str) -> float:
    canais = [v / 255 for v in _hex_rgb(cor)]
    linear = [v / 12.92 if v <= 0.04045 else ((v + 0.055) / 1.055) ** 2.4
              for v in canais]
    return 0.2126 * linear[0] + 0.7152 * linear[1] + 0.0722 * linear[2]


@tool
def avaliar_contraste(texto_hex: str, fundo_hex: str) -> dict:
    """Calcula a razão de contraste WCAG de duas cores HEX e verifica AA para texto normal e grande."""
    a, b = _luminancia(texto_hex), _luminancia(fundo_hex)
    razao = (max(a, b) + 0.05) / (min(a, b) + 0.05)
    return {
        "cor_texto": texto_hex.upper(),
        "cor_fundo": fundo_hex.upper(),
        "razao": round(razao, 4),
        "AA_texto_normal": razao >= 4.5,
        "AA_texto_grande": razao >= 3.0,
        "AAA_texto_normal": razao >= 7.0,
        "AAA_texto_grande": razao >= 4.5,
    }

# Testes locais independentes do Groq
assert avaliar_contraste.invoke({"texto_hex":"#000000", "fundo_hex":"#FFFFFF"})["AA_texto_normal"] is True
assert avaliar_contraste.invoke({"texto_hex":"#FFFFFF", "fundo_hex":"#FFFFFF"})["AA_texto_normal"] is False
print("Testes da ferramenta: OK")

Testes da ferramenta: OK


## 4. Estado compartilhado pelos agentes
O LangGraph transmite o resultado de cada etapa por meio de um dicionário tipado.

In [15]:
class Estado(TypedDict):
    texto_hex: str
    fundo_hex: str
    componente: str
    resultado_ferramenta: dict
    analise_auditor: str
    recomendacao_consultor: str

## 5. Agente 1 — Auditor de Acessibilidade
O agente aciona explicitamente a ferramenta `@tool`, depois interpreta o resultado. Assim, o cálculo é executado em toda avaliação, sem depender de o modelo decidir chamar a ferramenta.

In [16]:
def agente_auditor(estado: Estado) -> dict:
    # Uso efetivo da ferramenta pelo agente
    resultado = avaliar_contraste.invoke({
        "texto_hex": estado["texto_hex"],
        "fundo_hex": estado["fundo_hex"],
    })
    resposta = llm.invoke([
        SystemMessage(content=(
            "Você é um auditor técnico de acessibilidade web. "
            "Explique brevemente os resultados numéricos recebidos. "
            "Não recalcule, não invente critérios e não modifique os valores da ferramenta. "
            "Diferencie texto normal e grande. Responda em português."
        )),
        HumanMessage(content=(
            f"Componente: {estado['componente']}\n"
            f"Resultado da ferramenta: {resultado}\n"
            "Elabore um diagnóstico em até 130 palavras."
        )),
    ])
    return {"resultado_ferramenta": resultado, "analise_auditor": resposta.content}

## 6. Agente 2 — Consultor UX
O segundo agente **recebe** o resultado da ferramenta e o diagnóstico do primeiro. Sua responsabilidade é traduzir isso em orientação para a interface.

In [17]:
def agente_consultor(estado: Estado) -> dict:
    resposta = llm.invoke([
        SystemMessage(content=(
            "Você é um consultor de UX e acessibilidade. "
            "Leia o diagnóstico do auditor e proponha ações práticas. "
            "Não altere o valor de contraste nem diga que uma nova cor foi aprovada "
            "sem calculá-la. Não afirme que toda a interface é acessível com base apenas no contraste. "
            "Responda em português, de maneira objetiva."
        )),
        HumanMessage(content=(
            f"Componente: {estado['componente']}\n"
            f"Dados medidos: {estado['resultado_ferramenta']}\n"
            f"Diagnóstico do auditor: {estado['analise_auditor']}\n"
            "Produza duas recomendações práticas e uma conclusão curta."
        )),
    ])
    return {"recomendacao_consultor": resposta.content}

## 7. Montagem do grafo (colaboração entre agentes)

In [18]:
grafo = StateGraph(Estado)
grafo.add_node("auditor", agente_auditor)
grafo.add_node("consultor_ux", agente_consultor)
grafo.add_edge(START, "auditor")
grafo.add_edge("auditor", "consultor_ux")
grafo.add_edge("consultor_ux", END)
aplicacao = grafo.compile()
print("Grafo compilado: START → auditor → consultor_ux → END")

Grafo compilado: START → auditor → consultor_ux → END


## 8. Executar um exemplo
Edite as três variáveis para avaliar outras combinações. A cor de texto `#FFFFFF` sobre fundo `#777777` fica muito próxima, mas abaixo, do limiar de texto normal.

In [19]:
entrada = {
    "texto_hex": "#FFFFFF",
    "fundo_hex": "#777777",
    "componente": "Botão Continuar de uma aplicação web",
}

resultado_final = aplicacao.invoke(entrada)

print("===== FERRAMENTA =====")
print(resultado_final["resultado_ferramenta"])
print("\n===== AGENTE 1: AUDITOR =====")
print(resultado_final["analise_auditor"])
print("\n===== AGENTE 2: CONSULTOR UX =====")
print(resultado_final["recomendacao_consultor"])

===== FERRAMENTA =====
{'cor_texto': '#FFFFFF', 'cor_fundo': '#777777', 'razao': 4.4781, 'AA_texto_normal': False, 'AA_texto_grande': True, 'AAA_texto_normal': False, 'AAA_texto_grande': False}

===== AGENTE 1: AUDITOR =====
O botão “Continuar” apresenta texto branco (#FFFFFF) sobre fundo cinza médio (#777777). A razão de contraste medida é **4,48:1**.  

- **Texto normal** (até 18 pt ou 14 pt em negrito) não atinge o nível **AA** (exigência mínima 4,5:1) nem o nível **AAA** (exigência 7:1).  
- **Texto grande** (18 pt ou 14 pt em negrito) cumpre o critério **AA** (mínimo 3:1), mas ainda falha no **AAA** (exigência 4,5:1).  

Portanto, o botão está parcialmente acessível: atende apenas ao contraste requerido para texto grande em nível AA. Para garantir conformidade total, aumente o contraste – por exemplo, escurecendo o fundo ou clareando o texto – de modo que a razão alcance ≥ 4,5:1 para texto normal (AA) e ≥ 7:1 para AAA.

===== AGENTE 2: CONSULTOR UX =====
**Recomendação 1 – atender

## 9. Experimentos sugeridos

Execute a célula de entrada mais de uma vez usando: `#000000 / #FFFFFF` (alto contraste), `#FFFFFF / #FFFFFF` (nenhum contraste) e `#FFFFFF / #777777` (caso limítrofe). Compare os diagnósticos.

## 10. Roteiro breve para os slides

1. **Problema e domínio:** contraste de textos em interfaces web.
2. **Arquitetura:** usuário → Auditor (ferramenta) → Consultor UX → relatório.
3. **Demonstração:** entrada HEX, resultado numérico, duas respostas distintas.
4. **Conclusões e limitações:** colaboração, ferramenta determinística, papel do LLM; não avalia todos os critérios WCAG, e sugestões de cores exigem nova verificação.
5. **Slide final:** link do vídeo de apresentação, conforme o enunciado.

**Entrega solicitada:** arquivo `.py` ou `.ipynb` e slides em um único ZIP (além do vídeo linkado no slide final).